In [208]:
import pandas as pd
from pathlib import Path

outfield_dfs = []
gk_dfs = []

for season_dir in sorted(Path('data/47').iterdir()):
    out_path = season_dir / 'output' / 'outfield_players.parquet'
    gk_path  = season_dir / 'output' / 'goalkeepers.parquet'

    if out_path.exists():
        df = pd.read_parquet(out_path)
        df['season'] = season_dir.name
        outfield_dfs.append(df)
        print(f'Outfield {season_dir.name}: {df.shape[0]} rows x {df.shape[1]} cols')

    if gk_path.exists():
        df = pd.read_parquet(gk_path)
        df['season'] = season_dir.name
        gk_dfs.append(df)
        print(f'GK      {season_dir.name}: {df.shape[0]} rows x {df.shape[1]} cols')

    print()

outfield = pd.concat(outfield_dfs, ignore_index=True)
gk       = pd.concat(gk_dfs,       ignore_index=True)

print(f'Total outfield : {outfield.shape[0]:,} rows x {outfield.shape[1]} cols')
print(f'Total GK       : {gk.shape[0]:,} rows x {gk.shape[1]} cols')

Outfield 2021_2022: 5694 rows x 68 cols
GK      2021_2022: 564 rows x 53 cols

Outfield 2022_2023: 5278 rows x 68 cols
GK      2022_2023: 528 rows x 53 cols

Outfield 2023_2024: 5380 rows x 68 cols
GK      2023_2024: 538 rows x 51 cols

Outfield 2024_2025: 5478 rows x 68 cols
GK      2024_2025: 548 rows x 53 cols

Total outfield : 21,830 rows x 68 cols
Total GK       : 2,178 rows x 53 cols


In [209]:
# Separate the ID columns from the stat columns
ID_COLS = [
    'match_id', 'round', 'match_date', 'home_team', 'away_team',
    'player_id', 'player_name', 'team_id', 'team_name',
    'shirt_number', 'position_id', 'is_goalkeeper', 'season'
]

outfield_stat_cols = [c for c in outfield.columns if c not in ID_COLS]
gk_stat_cols       = [c for c in gk.columns if c not in ID_COLS]

print(f'Outfield stat columns ({len(outfield_stat_cols)}):')
for c in outfield_stat_cols:
    print(f'  {c}')

print(f'\nGK stat columns ({len(gk_stat_cols)}):')
for c in gk_stat_cols:
    print(f'  {c}')

Outfield stat columns (55):
  ShotsOffTarget
  ShotsOnTarget
  accurate_passes
  assists
  chances_created
  defensive_actions
  expected_assists
  expected_goals
  expected_goals_on_target_variant
  goals
  minutes_played
  rating_title
  shot_accuracy
  xg_and_xa
  blocked_shots
  big_chance_created_team_title
  errors_led_to_goal
  conceded_penalties
  penalties_won
  owngoal
  missed_penalty
  accurate_crosses
  corners
  dispossessed
  dribbles_succeeded
  expected_goals_non_penalty
  long_balls_accurate
  passes_into_final_third
  touches
  touches_opp_box
  Offsides
  big_chance_missed_title
  shots_woodwork
  clearances
  dribbled_past
  headed_clearance
  interceptions
  matchstats.headers.tackles
  recoveries
  shot_blocks
  last_man_tackle
  clearance_off_the_line
  aerials_won
  duel_lost
  duel_won
  fouls
  ground_duels_won
  was_fouled
  accurate_crosses_total
  accurate_passes_total
  aerials_won_total
  dribbles_succeeded_total
  ground_duels_won_total
  long_balls_acc

In [210]:
print('=== OUTFIELD NULL RATES ===')
null_rates = outfield[outfield_stat_cols].isnull().mean().sort_values(ascending=False)
print(null_rates[null_rates > 0].to_string())

print('\n=== GK NULL RATES ===')
gk_null_rates = gk[gk_stat_cols].isnull().mean().sort_values(ascending=False)
print(gk_null_rates[gk_null_rates > 0].to_string())

print('\n=== OUTFIELD BASIC STATS (non-null counts + mean) ===')
print(outfield[outfield_stat_cols].describe().loc[['count','mean','min','max']].T.to_string())

print('\n=== GK BASIC STATS (non-null counts + mean) ===')
print(gk[gk_stat_cols].describe().loc[['count','mean','min','max']].T.to_string())

=== OUTFIELD NULL RATES ===
missed_penalty                      0.997801
owngoal                             0.995419
penalties_won                       0.989647
conceded_penalties                  0.987723
clearance_off_the_line              0.986441
errors_led_to_goal                  0.986349
last_man_tackle                     0.983326
shots_woodwork                      0.976042
Offsides                            0.892442
big_chance_missed_title             0.890380
big_chance_created_team_title       0.870454
corners                             0.831562
blocked_shots                       0.736143
expected_goals_on_target_variant    0.724645
shot_accuracy                       0.525424
shot_accuracy_total                 0.525424
accurate_crosses_total              0.473981
accurate_crosses                    0.473981
headed_clearance                    0.462345
was_fouled                          0.455153
expected_goals_non_penalty          0.425286
dribbles_succeeded         

In [ ]:
# Auto-drop above 50% null
drop_candidates = null_rates[null_rates > 0.50].index.tolist()

# Keep blocked_shots despite >50% null — min value is 1, real event, not redundant
drop_candidates = [c for c in drop_candidates if c != 'blocked_shots']

# Add redundant columns manually
REDUNDANT = ['xg_and_xa', 'expected_goals_non_penalty', 'duel_won']
drop_candidates = list(set(drop_candidates + REDUNDANT))

print(f"Dropping {len(drop_candidates)} columns:")
for c in sorted(drop_candidates):
    print(f"  {c}")

# Create cleaned dataframe
outfield_clean = outfield.drop(columns=drop_candidates)

print(f"\noutfield shape before : {outfield.shape}")
print(f"outfield shape after  : {outfield_clean.shape}")

Dropping 18 columns:
  Offsides
  big_chance_created_team_title
  big_chance_missed_title
  clearance_off_the_line
  conceded_penalties
  corners
  duel_won
  errors_led_to_goal
  expected_goals_non_penalty
  expected_goals_on_target_variant
  last_man_tackle
  missed_penalty
  owngoal
  penalties_won
  shot_accuracy
  shot_accuracy_total
  shots_woodwork
  xg_and_xa

outfield shape before : (21830, 68)
outfield shape after  : (21830, 50)


In [212]:
# Auto-drop above 50% null
gk_drop_candidates = gk_null_rates[gk_null_rates > 0.50].index.tolist()

# Keep aerials_won and aerials_won_total despite >50% null — sweeper keeper behavior
gk_drop_candidates = [c for c in gk_drop_candidates if c not in ['aerials_won', 'aerials_won_total']]

# Keep ground_duels_won and ground_duels_won_total despite >50% null — same reason
gk_drop_candidates = [c for c in gk_drop_candidates if c not in ['ground_duels_won', 'ground_duels_won_total']]

# Add near-zero and redundant columns manually
REDUNDANT_GK = ['interceptions']
gk_drop_candidates = list(set(gk_drop_candidates + REDUNDANT_GK))

print(f"Dropping {len(gk_drop_candidates)} columns:")
for c in sorted(gk_drop_candidates):
    print(f"  {c}")

# Create cleaned dataframe
gk_clean = gk.drop(columns=gk_drop_candidates)

print(f"\ngk shape before : {gk.shape}")
print(f"gk shape after  : {gk_clean.shape}")

Dropping 13 columns:
  assists
  chances_created
  conceded_penalties
  dribbles_succeeded
  dribbles_succeeded_total
  duel_lost
  duel_won
  errors_led_to_goal
  expected_assists
  interceptions
  saved_penalties
  was_fouled
  xg_and_xa

gk shape before : (2178, 53)
gk shape after  : (2178, 40)


In [213]:
# Replace 0.01 placeholders
PLACEHOLDER_COLS_OUTFIELD = ['expected_goals', 'expected_assists']
PLACEHOLDER_COLS_GK       = ['expected_goals_on_target_faced']

for col in PLACEHOLDER_COLS_OUTFIELD:
    if col in outfield_clean.columns:
        mask = outfield_clean[col] == 0.01
        outfield_clean.loc[mask, col] = 0
        print(f'Outfield {col:<35} replaced {mask.sum()} placeholder 0.01s')

for col in PLACEHOLDER_COLS_GK:
    if col in gk_clean.columns:
        mask = gk_clean[col] == 0.01
        gk_clean.loc[mask, col] = 0
        print(f'GK      {col:<35} replaced {mask.sum()} placeholder 0.01s')

Outfield expected_goals                      replaced 306 placeholder 0.01s
Outfield expected_assists                    replaced 4563 placeholder 0.01s
GK      expected_goals_on_target_faced      replaced 2 placeholder 0.01s


In [214]:
# Drop null rating_title BEFORE fillna so we don't fill it with 0
outfield_before = len(outfield_clean)
gk_before       = len(gk_clean)
outfield_clean  = outfield_clean.dropna(subset=['rating_title'])
gk_clean        = gk_clean.dropna(subset=['rating_title'])
print(f'\nDropped {outfield_before - len(outfield_clean)} outfield rows with null rating_title')
print(f'Dropped {gk_before - len(gk_clean)} GK rows with null rating_title')


Dropped 26 outfield rows with null rating_title
Dropped 0 GK rows with null rating_title


In [215]:
# Rename dirty column BEFORE computing stat cols
outfield_clean = outfield_clean.rename(columns={'matchstats.headers.tackles': 'tackles'})
gk_clean       = gk_clean.rename(columns={'matchstats.headers.tackles': 'tackles'})

# Recompute stat cols AFTER rename so the list is accurate
outfield_stat_cols = [c for c in outfield_clean.columns if c not in ID_COLS]
gk_stat_cols       = [c for c in gk_clean.columns if c not in ID_COLS]

In [216]:
outfield_nulls_before = outfield_clean[outfield_stat_cols].isnull().sum()
gk_nulls_before       = gk_clean[gk_stat_cols].isnull().sum()

# Fill remaining nulls with 0
outfield_clean[outfield_stat_cols] = outfield_clean[outfield_stat_cols].fillna(0)
gk_clean[gk_stat_cols]             = gk_clean[gk_stat_cols].fillna(0)

print('\n=== OUTFIELD — nulls filled with 0 ===')
outfield_filled = outfield_nulls_before[outfield_nulls_before > 0].sort_values(ascending=False)
for col, count in outfield_filled.items():
    pct = count / len(outfield_clean) * 100
    print(f'  {col:<40} {count:>5} rows filled ({pct:.1f}%)')
print(f'  Total: {outfield_filled.sum():,} values filled across {len(outfield_filled)} columns')

print('\n=== GK — nulls filled with 0 ===')
gk_filled = gk_nulls_before[gk_nulls_before > 0].sort_values(ascending=False)
for col, count in gk_filled.items():
    pct = count / len(gk_clean) * 100
    print(f'  {col:<40} {count:>5} rows filled ({pct:.1f}%)')
print(f'  Total: {gk_filled.sum():,} values filled across {len(gk_filled)} columns')



=== OUTFIELD — nulls filled with 0 ===
  blocked_shots                            16044 rows filled (73.6%)
  accurate_crosses_total                   10323 rows filled (47.3%)
  accurate_crosses                         10323 rows filled (47.3%)
  headed_clearance                         10067 rows filled (46.2%)
  was_fouled                                9910 rows filled (45.5%)
  dribbles_succeeded                        9256 rows filled (42.5%)
  dribbles_succeeded_total                  9256 rows filled (42.5%)
  ShotsOffTarget                            9216 rows filled (42.3%)
  ShotsOnTarget                             9216 rows filled (42.3%)
  expected_goals                            9212 rows filled (42.2%)
  long_balls_accurate_total                 4132 rows filled (19.0%)
  long_balls_accurate                       4132 rows filled (19.0%)
  expected_assists                          3395 rows filled (15.6%)
  passes_into_final_third                   2055 rows filled (9

In [217]:
# Filter minimum minutes < 30 to remove players with limited playing time that could skew the stats
outfield_before = len(outfield_clean)
gk_before       = len(gk_clean)

outfield_clean = outfield_clean[outfield_clean['minutes_played'] >= 30]
gk_clean       = gk_clean[gk_clean['minutes_played'] >= 30]

print(f'Outfield rows removed (< 30 mins): {outfield_before - len(outfield_clean):>5} ({(outfield_before - len(outfield_clean)) / outfield_before * 100:.1f}%)')
print(f'GK rows removed (< 30 mins)      : {gk_before - len(gk_clean):>5} ({(gk_before - len(gk_clean)) / gk_before * 100:.1f}%)')
print(f'\nOutfield remaining : {len(outfield_clean):,} rows')
print(f'GK remaining       : {len(gk_clean):,} rows')

Outfield rows removed (< 30 mins):   164 (0.8%)
GK rows removed (< 30 mins)      :     0 (0.0%)

Outfield remaining : 21,640 rows
GK remaining       : 2,178 rows


In [218]:
print(f'\nOutfield final shape : {outfield_clean.shape}')
print(f'GK final shape       : {gk_clean.shape}')
print(f'\nAny nulls in outfield stat cols : {outfield_clean[outfield_stat_cols].isnull().any().any()}')
print(f'Any nulls in GK stat cols       : {gk_clean[gk_stat_cols].isnull().any().any()}')
print(f'\nOutfield rating_title zeros : {(outfield_clean["rating_title"] == 0).sum()}')
print(f'GK rating_title zeros       : {(gk_clean["rating_title"] == 0).sum()}')
print(f'\nOutfield rating_title stats:\n{outfield_clean["rating_title"].describe()}')


Outfield final shape : (21640, 50)
GK final shape       : (2178, 40)

Any nulls in outfield stat cols : False
Any nulls in GK stat cols       : False

Outfield rating_title zeros : 0
GK rating_title zeros       : 0

Outfield rating_title stats:
count    21640.000000
mean         7.009238
std          0.802556
min          3.660000
25%          6.467500
50%          7.000000
75%          7.530000
max          9.750000
Name: rating_title, dtype: float64


In [219]:
from pathlib import Path

PROCESSED_DIR = Path('data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Export cleaned dataframes
outfield_clean.to_parquet(PROCESSED_DIR / 'outfield_clean.parquet', index=False)
gk_clean.to_parquet(PROCESSED_DIR / 'gk_clean.parquet', index=False)

outfield_clean.to_csv(PROCESSED_DIR / 'outfield_clean.csv', index=False)
gk_clean.to_csv(PROCESSED_DIR / 'gk_clean.csv', index=False)

print(f'Saved to {PROCESSED_DIR.resolve()}')
print(f'  outfield_clean.parquet — {len(outfield_clean):,} rows x {outfield_clean.shape[1]} cols')
print(f'  gk_clean.parquet       — {len(gk_clean):,} rows x {gk_clean.shape[1]} cols')

Saved to C:\Users\natmaw\NatMaws Documents\Boston Stuff\EECE 5644 Machine Learning and Pattern Recognition\Final Project\GOALS-Game_Outcome_and_Analytics_Learning_System-\data\processed
  outfield_clean.parquet — 21,640 rows x 50 cols
  gk_clean.parquet       — 2,178 rows x 40 cols
